<a href="https://colab.research.google.com/github/Noman654/dataengineer_prep/blob/main/pyspark/window_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪟 Window Functions — The Consecutive Months Problem

---

### 💬 New message from Jen (Marketing)

> **Jen** — Monday 9:14 AM
>
> hey!! 👋 quick favor. marketing wants to launch a "thank you for sticking with us" voucher this week.
>
> I need a list of our **loyal regulars** — customers who bought something in **3+ consecutive months**. finance thinks these are our lowest-churn-risk folks and we want to lock them in before summer.
>
> can you pull the list by EOD today? 🙏 you're the best
>
> (ps. walk-ins don't count obviously, only loyalty members with customer_id)

---

Okay. You just opened Slack to this. Let's solve it.

## 🎯 What you'll learn

- How to think in **window functions** instead of self-joins
- The `lag()` pattern for detecting sequences over time
- The classic **gaps and islands** problem — and why it shows up in interviews constantly
- How to answer "N consecutive anything" questions in a handful of lines

**Prerequisites:** You've seen basic `groupBy` and `agg` before.

**Difficulty:** 🟡 Intermediate

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("zephyr_window_functions").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

## 📦 The data

For this notebook we'll use a tiny slice of Zephyr's `transactions` table — 6 customers, 17 transactions, 5 months. Small enough that you can reason about the right answer in your head before we touch any code.

*(In the real repo, this loads from `assets/sample_data/transactions.parquet`. Here it's inline so the notebook is self-contained.)*

In [ ]:
sample_transactions = [
    # Alice (101) — bought every month Jan→May (our clearest regular)
    ("tx_001", 101, "2024-01-15", 12.50),
    ("tx_002", 101, "2024-02-03", 8.75),
    ("tx_003", 101, "2024-03-22", 15.00),
    ("tx_004", 101, "2024-04-11", 9.25),
    ("tx_005", 101, "2024-05-08", 11.00),

    # Bob (102) — Jan, Feb, skipped Mar, Apr — NOT 3 consecutive
    ("tx_006", 102, "2024-01-20", 7.50),
    ("tx_007", 102, "2024-02-14", 8.00),
    ("tx_008", 102, "2024-04-02", 10.25),

    # Carol (103) — Feb, Mar, Apr — exactly 3 consecutive ✓
    ("tx_009", 103, "2024-02-05", 6.50),
    ("tx_010", 103, "2024-03-18", 12.00),
    ("tx_011", 103, "2024-04-27", 9.75),

    # Dave (104) — one transaction, one month
    ("tx_012", 104, "2024-03-10", 5.50),

    # Eve (105) — two transactions, both in March — same month twice
    ("tx_013", 105, "2024-03-05", 8.00),
    ("tx_014", 105, "2024-03-25", 11.50),

    # Frank (106) — Jan, Feb, Mar then nothing — exactly 3 ✓
    ("tx_015", 106, "2024-01-08", 7.00),
    ("tx_016", 106, "2024-02-19", 9.50),
    ("tx_017", 106, "2024-03-30", 13.25),

    # Gina (107) — Jan, Feb, Mar (3 consecutive ✓) BUT spend is dropping ↓
    # She's a regular AND a churn risk — matches both Jen's queries
    ("tx_018", 107, "2024-01-12", 25.00),
    ("tx_019", 107, "2024-02-09", 15.00),
    ("tx_020", 107, "2024-03-14",  8.00),
]

transactions = (
    spark.createDataFrame(
        sample_transactions,
        ["tx_id", "customer_id", "ts", "total_amount"],
    )
    .withColumn("ts", F.to_timestamp("ts"))
)

transactions.show()

### 🧠 Before writing any code — figure out the answer by hand

Seriously, pause for 30 seconds. Which customers should end up on Jen's list?

<details>
<summary>Click for the expected answer</summary>

- **Alice (101)** — bought every month Jan → May ✓
- **Carol (103)** — bought Feb, Mar, Apr ✓
- **Frank (106)** — bought Jan, Feb, Mar ✓
- **Gina (107)** — bought Jan, Feb, Mar ✓

Bob skipped March. Dave bought once. Eve bought twice but in the same month.

Keep this in your head — when your query runs, if it returns anything else, you know it's wrong.

</details>

## 🤔 Pause. How would you even do this?

Before scrolling down, actually think about it. How do you find "bought something in **3 consecutive months**"?

<details>
<summary>Hint 1</summary>

You can't do this with `groupBy` alone. `groupBy(customer_id).count()` tells you *how many* months they bought in, not whether those months were *consecutive*.
</details>

<details>
<summary>Hint 2</summary>

You need each row to "know" what came before it. That's exactly what **window functions** do — they let a row peek at its neighbors inside a sorted group.
</details>

<details>
<summary>Hint 3</summary>

Key function: `F.lag(col, n)` — returns the value of `col` from `n` rows earlier *within a window*. If you sort transactions by month per customer, `lag(month, 1)` gives you the previous month they bought in.
</details>

## 💡 Walkthrough

### Step 1 — Reduce to one row per customer per month

We don't care how many times Alice bought in January, only *that* she did. Collapse down:

In [ ]:
monthly = (
    transactions
    .withColumn("month", F.date_trunc("month", "ts"))
    .select("customer_id", "month")
    .distinct()
    .orderBy("customer_id", "month")
)

monthly.show()

Eve's two March transactions collapsed to one row. Good — that's what `.distinct()` did.

### Step 2 — For each row, what was the previous month?

Here's where windows come in. We define a window that says *"group by customer_id, and within each group, sort by month"*:

```python
w = Window.partitionBy("customer_id").orderBy("month")
```

Now any window function (`lag`, `lead`, `row_number`, `sum`, `rank`...) applied with `.over(w)` operates **inside that sorted group**. `lag(month, 1).over(w)` = *"for each row, what was `month` one row ago, within this customer's sorted timeline?"*

In [ ]:
w = Window.partitionBy("customer_id").orderBy("month")

with_prev = monthly.withColumn("prev_month", F.lag("month", 1).over(w))

with_prev.show()

Look at Alice's rows. Row by row, `prev_month` now tells her *exactly* what month she bought in just before this one. The first row of each customer has `prev_month = null` because there's nothing before it — that's normal.

### Step 3 — Is each row consecutive with the previous?

If `month - prev_month` is exactly 1 month → consecutive. Otherwise → gap.

In [ ]:
with_flag = with_prev.withColumn(
    "is_consecutive",
    F.when(F.months_between("month", "prev_month") == 1, 1).otherwise(0),
)

with_flag.show()

### Step 4 — The trick: spotting a streak of 3

Here's the clever bit. `is_consecutive = 1` means "this month touches the previous one". But that's only a **streak of 2**. To find a streak of **3 consecutive months**, we need a row where `is_consecutive = 1` **and** the row just before *also* had `is_consecutive = 1`.

In other words: lag the flag itself.

In [ ]:
final = (
    with_flag
    .withColumn("prev_consecutive", F.lag("is_consecutive", 1).over(w))
    .filter((F.col("is_consecutive") == 1) & (F.col("prev_consecutive") == 1))
    .select("customer_id")
    .distinct()
)

final.show()

## 📤 Ship it to Jen

```
customer_id
-----------
101   (Alice)
103   (Carol)
106   (Frank)
```

Exactly what we predicted by hand. Copy, paste into Slack, mission accomplished.

> **You** → Jen: here's the list, 3 customers fit. LMK if you want it broader (2 consecutive months) or different time window 👍
>
> **Jen**: omg lifesaver ty 🙌

---

### 🧠 The pattern you just learned

This "lag-and-flag" approach solves a **huge** family of problems:

- 3 consecutive months of purchases ✓
- Login streaks (daily active users N days in a row)
- Price drops on N consecutive days
- "First time a user did X after doing Y"
- Session detection (gap of >30 min = new session)

All the same pattern: **sort by time, lag the relevant column, compare neighbors.** Once you see it, you see it everywhere.

## 🎯 Your turn — Jen came back

> **Jen** — 2:30 PM
>
> ok different angle. finance is worried about **churn**. can you find customers whose **monthly spending dropped for 3 months in a row**? like Jan = $50, Feb = $30, Mar = $15. those are the ones we want to save before they leave us entirely.

Same dataset. Same window trick. Different comparison. (Hint: you're now comparing *amounts* across months, not *existence* of months.)

Try it in the cell below. No solution at the bottom of the notebook — that's deliberate. Struggle with it for 10 minutes before you look anything up.

In [ ]:
# your code here




## 🏆 Boss Level — Marcus wants top customers per month

> **Marcus (CFO)** — Friday 6:04 PM (of course)
>
> Need the top **2 customers by total spend for each month** of 2024. Executive deck Monday. The board wants to see who our biggest spenders are, month by month.

This one needs `row_number()` or `rank()` or `dense_rank()` — and the difference between them **actually matters here**.

- `row_number()` — unique sequential numbers, even for ties → (1, 2, 3, 4)
- `rank()` — ties share a rank, then skip → (1, 2, 2, 4)
- `dense_rank()` — ties share a rank, no skip → (1, 2, 2, 3)

**Think before you code:** if two customers tie for #2 in January, which function gives Marcus the answer he probably wants? Would he rather see 2 customers or 3 in his "top 2"? There's no single right answer — the point is to *notice the ambiguity and ask* before shipping.

**The mental move:** in the walkthrough, your window was `partitionBy("customer_id")`. Here you need to flip it — partition by **month**, order by **spend descending**. Same tool, different question.

Try it in the cell below before scrolling to the solution.

In [ ]:
# your code here



---

## 💡 Solutions

> ⚠️ **Spoilers below.** Don't scroll any further until you've actually tried both exercises above. Struggling with them for 10 minutes is worth more than reading the solution in 30 seconds.
>
> Still here? OK, let's walk through both.

### 💡 Solution 1 — Jen's churn query

**The ask:** find customers whose spending dropped for 3 months in a row.

The pattern is almost identical to the consecutive-months query — but instead of checking whether months *touch*, we check whether **spend values decrease**.

**Steps:**
1. Aggregate to monthly spend per customer (`sum(total_amount)`)
2. Use the same `Window.partitionBy("customer_id").orderBy("month")`
3. Use `lag(spend, 1)` for last month's spend, and `lag(spend, 2)` for the month before that
4. Filter for rows where **current < previous** AND **previous < previous-previous**
5. Return distinct `customer_id`s

Two lags in one query — that's the new move.

In [ ]:
monthly_spend = (
    transactions
    .withColumn("month", F.date_trunc("month", "ts"))
    .groupBy("customer_id", "month")
    .agg(F.sum("total_amount").alias("spend"))
)

w = Window.partitionBy("customer_id").orderBy("month")

churn_risks = (
    monthly_spend
    .withColumn("prev_spend",      F.lag("spend", 1).over(w))
    .withColumn("prev_prev_spend", F.lag("spend", 2).over(w))
    .filter(
        (F.col("spend")      < F.col("prev_spend")) &
        (F.col("prev_spend") < F.col("prev_prev_spend"))
    )
    .select("customer_id")
    .distinct()
)

churn_risks.show()

**Expected output:** `customer_id = 107` (Gina, whose spend went $25 → $15 → $8).

**Gotcha you might have hit:** if you only used `lag(spend, 1)` and checked `spend < prev_spend`, you'd find *one-month* drops, not a 3-month decline. You need to look back **two** positions to establish a trend.

**Alternative approach:** instead of double-lagging, you could use `lag(spend, 1)` and then apply the same lag-the-flag trick from Step 4 of the walkthrough. Same answer, different mental model. Both are valid — the double-lag is more direct for *exactly* 3 months; the flag trick scales cleanly to "N months".

**Bonus insight:** Gina also appeared in Jen's *first* list (3 consecutive months of purchases). She's both a loyal regular AND a churn risk — she's still buying every month but her spend is collapsing. In a real job, you'd ping Jen back: *"Hey, Gina is on both lists. She's probably your highest-priority save — still engaged, but bleeding spend fast."* That kind of cross-cut is the difference between "running a query" and "doing analytics".

### 💡 Solution 2 — Marcus's top 2 customers per month

**The ask:** top 2 customers by total spend for each month of 2024.

**The pattern:**
1. Aggregate to customer-month total spend (reuse `monthly_spend` from above)
2. Window **partitioned by month** (not customer!) and ordered by spend descending
3. Apply a ranking function (`row_number`, `rank`, or `dense_rank`) over that window
4. Filter `rank ≤ 2`

The partition-key flip — from "per customer" to "per month" — is the *whole* mental move here. Windows always let you ask *"within this group, what's the order?"*. Changing the group changes the question.

In [ ]:
# Reuse monthly_spend from the churn solution
monthly_spend = (
    transactions
    .withColumn("month", F.date_trunc("month", "ts"))
    .groupBy("customer_id", "month")
    .agg(F.sum("total_amount").alias("spend"))
)

# Flip the window: partition by MONTH, order by spend descending
month_window = Window.partitionBy("month").orderBy(F.col("spend").desc())

top_2_per_month = (
    monthly_spend
    .withColumn("rn", F.row_number().over(month_window))
    .filter(F.col("rn") <= 2)
    .orderBy("month", "rn")
)

top_2_per_month.show()

**Why `row_number()` here?** It guarantees exactly 2 rows per month — no ties, no surprises. Executives generally want a fixed-size answer for their slide.

**When would you pick the others?**

- **`rank()`** — if ties should both appear, and the next rank is skipped. Two customers tied at #1 both get rank 1, then the next is rank 3. "Top 2 with `rank()`" might return 1 row or 3 rows, depending on ties.
- **`dense_rank()`** — same as `rank` but no skip. Two #1s, then a #2. "Top 2 with `dense_rank()`" might return more than 2 rows (everyone at rank 1 + everyone at rank 2).

**The real judgment call:** if two customers tie exactly in January, do you arbitrarily pick one (`row_number`), show both and skip ahead (`rank`), or show both without skipping (`dense_rank`)? **There's no universal right answer.** The point of this exercise is to *notice the ambiguity and ask Marcus what he actually wants* before shipping. That's the difference between a junior and a senior engineer — not the SQL, the question.

**Scaling note:** on 500K rows, this runs fine on a single machine. On 500M rows across 200 stores, you'd want the data partitioned by month (or month + hash of customer_id) so the window doesn't shuffle the entire table. Partition once, query many times.

## 📚 Further reading (later, not now)

- [Spark docs — Window functions](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html)
- *Gaps and Islands* by Itzik Ben-Gan — the canonical essay on this pattern
- Databricks blog on window function performance — matters once you're past ~10M rows

That's enough window functions for one day. Save your notebook. Go get a coffee (Zephyr brand, obviously).

---

## 🧠 Now test yourself

Walking through a notebook is one thing. Explaining the concepts **out loud, under pressure** is another — and that's what interviews actually feel like.

**→ [`quiz/window_functions.md`](quiz/window_functions.md)** — 10 self-check questions (🟢 basics → ⚡ senior judgment) with collapsible answers. Say each answer out loud before you peek.

If you can answer 8+ without looking, you're ready for the window-functions portion of any DE interview.